# Solution R8 · Warm-up: from the tree to the forest

**Type**: Implement with scikit-learn

## Objective

Compare in cross-validation a single tree and a forest of 200 trees, check that the **out-of-bag** error estimates the test one well and read the three most important variables.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.tree import DecisionTreeClassifier

data = load_breast_cancer(as_frame=True)
X, y = data.data, (data.target == 0).astype(int)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)

## Commented solution

In [ ]:
tree_scores = cross_val_score(DecisionTreeClassifier(random_state=0), X, y, cv=cv)
forest_scores = cross_val_score(RandomForestClassifier(200, random_state=0, n_jobs=-1), X, y, cv=cv)
tree_acc, tree_sd = float(tree_scores.mean()), float(tree_scores.std())
forest_acc, forest_sd = float(forest_scores.mean()), float(forest_scores.std())
gain = forest_acc - tree_acc
print(f"tree {tree_acc:.3f} +/- {tree_sd:.3f} | forest {forest_acc:.3f} +/- {forest_sd:.3f} | gain {gain:.3f}")

In [ ]:
forest = RandomForestClassifier(300, oob_score=True, random_state=0, n_jobs=-1).fit(X_tr, y_tr)
oob, test_acc = float(forest.oob_score_), float(forest.score(X_te, y_te))
print(f"OOB accuracy {oob:.3f} | test accuracy {test_acc:.3f}")

In [ ]:
importances = pd.Series(forest.feature_importances_, index=X.columns).sort_values(ascending=False)
top3 = importances.index[:3].tolist()
print(importances.head(5).round(3).to_string())

## Check

In [ ]:
with verify():
    check(0.90 < tree_acc < 0.95 and 0.95 < forest_acc < 0.99, "tree between 0.90 and 0.95, forest between 0.95 and 0.99", "cross_val_score(..., cv=cv).mean()")
    check(gain > 0.02 and forest_sd < tree_sd + 0.01, "the forest gains over 2 points and is not more variable than the tree", "forest_acc - tree_acc")
    check(abs(oob - test_acc) < 0.03 and oob > 0.94, "the OOB accuracy (over 0.94) is within 3 points of the test one: a free validation", "forest.oob_score_")
    check(len(top3) == 3 and sum("concave points" in v for v in top3) == 2, "among the three most important variables appear the two `concave points` measurements (`mean` and `worst`)", "np.argsort(-forest.feature_importances_)[:3]")
done()

## What you learned

- **Averaging trees works:** in cross-validation the single tree has accuracy 0.926 and the forest 0.963 (+3.7 points), with a similar or smaller variability across folds (0.022 against 0.017). No optimization: 200 trees with the default parameters.
- **The OOB is a free validation:** each tree has not seen about a third of the examples, and the average of the "out-of-sample" predictions gives an accuracy (0.957) very close to the one measured on the test set (0.942). With independent data you can avoid keeping a validation set aside; with duplicate or grouped rows you cannot (module 8, exercise C1).
- **Impurity importances have to be taken with caution:** the highest are `mean concave points` (0.146), `worst concave points` (0.136) and `worst perimeter` (0.107), but right after come `worst radius` (0.105) and `worst area` (0.100), which measure almost the same thing as the perimeter: with highly correlated variables the importance is split among the copies. For a reliable judgment you use the permutation on the test set (lesson 8.2).

## Common pitfall

Searching for the "right" number of trees with a grid: more trees never hurt (the error settles), you choose a number high enough and pay in time.

## Interview question

<details><summary>What is the out-of-bag error and when is it not reliable?</summary>

It is the error computed, for each example, using only the trees that did not see it in their bootstrap sample: a validation estimate without a separate set. It is not reliable if the examples are not independent (duplicate rows, repeated measurements, time series), because a copy of the example can be in the sample of the tree.

</details>